# Create DSL search queries

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Literal

from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [ ]:
class JobProfile(BaseModel):
    years_of_experience: Literal["0-2", "3-5", "6-9", "10+"] = Field(
        description="Experience level of the job profile",
    )
    skills: list[str] = Field(
        description="List of key technical and domain skills relevant to the job profile",
    )
    position_title: list[str] = Field(
        description="List of suitable job titles and role terms for the job profile",
    )
    languages: list[str] = Field(
        description="Spoken languages relevant to the job profile",
    )
    location: str = Field(
        description="Location relevant to the job profile",
    )
    education: list[str] = Field(
        description="List of suitable study programs/degrees for the job profile",
    )

In [ ]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.7-flash",
#     temperature=1.0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )

# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore
# dsl_gen = llm.with_structured_output(
#     DSLQuery,
#     method="function_calling",
#     include_raw=True,
# )

# llm = ChatOpenAI(model="gpt-5.6-sol")
llm = ChatOpenAI(model="gpt-5.6-terra")
structured = llm.with_structured_output(
    JobProfile,
    method="json_schema",
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
system_prompt = f"""
Create a job profile for the job description

## Guidelines
- position_title:
    - List of terms that can be found in job titles
    - Do not list more than 8 items
    - The terms should not be redundant
    - The list should contain common ranks (single item like "Director" or "Associate")
    - Common short titles like "Data Scientist" or "Consultant"
    - Prefer single words over multi-word titles
    - Avoid redundancies, few (1-2) two word titles are acceptable like "Data Scientist" or "Product Manager"
- education:
    - List suitable study programs for the job profile, do not list the degree level
""".strip()

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

raw = structured.invoke(messages)
job_profile = JobProfile.model_validate(raw)

In [ ]:
print(json.dumps(job_profile.model_dump(), indent=2))

In [ ]:
stop

## Create DSL Queries

In [ ]:
docs = Path("..") / "docs"
multi_source_dsl_path = (
    docs / "coresignal" / "Employee APIs" / "Multi-source Employee API.json"
)

with open(multi_source_dsl_path, "r") as file:
    multi_source_dsl_json = file.read()

In [ ]:
class DSLQuery(BaseModel):
    query: dict[str, Any] = Field(
        description="Top level query object for the DSL query"
    )

In [ ]:
system_prompt = f"""
Task:
- Generate a DSL search query in JSON format that finds candidates matching the job description.
- The query should not be overly complex, so that it can still be executable by the Coresignal API.

Required Filtering:
- General: Filter for is_working = true and is_deleted = false and is_parent = true
- Language:
    - Language skills should not be "Elementary proficiency" or "Limited working proficiency"
    - Match the languages in english and the native language (e.g. "English" and "Englisch")
- Match the years of experience against the total_experience_duration_months
- Exclude candidates from India

Scoring (descending order of importance):
- Region: Local candidates are better than remote candidates.
- Education title should match the education list in the job profile --> More matches better
- Search position_title under experience --> More matches better

DSL Query JSON Schema:
```json
{multi_source_dsl_json}
```
""".strip()

In [ ]:
# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore
# dsl_gen = llm.with_structured_output(
#     DSLQuery,
#     method="function_calling",
# )

llm = ChatOpenAI(model="gpt-5.6-terra")
dsl_gen = llm.with_structured_output(
    DSLQuery,
    method="json_mode",
)

In [ ]:
prompt_mk = f"""
Job description:
{json.dumps(job_profile.model_dump(), indent=2)}
"""

query_raw = dsl_gen.invoke(
    [
        ("system", system_prompt),
        (
            "human",
            prompt_mk,
        ),
    ]
)
query = DSLQuery.model_validate(query_raw)
print(json.dumps(query.model_dump(), indent=2))